# Agentic coding: working effectively with AI

Today is different from earlier lectures. We are *not* learning new Python syntax.
We are learning how to work with the tools that now write most code, AI coding
assistants, and, just as importantly, how to not be fooled by them.

**Topics**
1. How coding has changed, and why the last lectures still matter
2. What a large language model actually is (a useful mental model)
3. Chat assistants vs. agents
4. The human-in-the-loop workflow: specify → generate → read → run → test
5. Failure modes: hallucinated APIs, plausible-but-wrong logic, silent data errors
6. Verifying generated code
7. Reproducibility, provenance, and academic integrity
8. Live demo: Claude Code doing the heavy lifting
9. Today's exercise

# 1. Coding has changed

- Most code today is written **with AI assistance**, not typed from memory.
- The valuable skill is no longer *recalling syntax*. It is:
  - **specifying** a problem clearly,
  - **reading** and **running** what you get back,
  - **verifying** that it is actually correct.
- This is exactly why Lectures 1–3 matter. You cannot judge code you cannot read.
  - Variables, loops, functions, types (L1–L2) → you can *read* the output.
  - The command line (L3) → assistants live there too.
- The assistant is a fast, confident junior colleague who is sometimes wrong and
  never tells you when. **You** are the senior scientist who signs off.

# 2. What is a large language model (LLM)?

The useful mental model, in one sentence:

> An LLM is a system that, given some text, predicts what text is likely to come
> next (trained based on an enormous amount of writing).

That's it. It is autocomplete on steroids.

- It was trained by reading a huge fraction of the internet, books, and code, and
  learning the *patterns* in that text.
- It is **not** a database, and **not** a search engine. It does not "look up"
  facts. It *generates* the most plausible continuation.
- This one fact explains almost everything it does well and badly.

## A toy version you can run

Below is a tiny "language model": from a few sentences it learns which word tends
to follow which, then generates new text. A real LLM is this same idea, namely
predict-the-next-piece, scaled up by a factor of billions and made far smarter
about *context*.

In [ ]:
import random
from collections import defaultdict, Counter
random.seed(1)

training_text = (
    "the penguin eats fish . the penguin swims in the sea . "
    "the seal eats the penguin . the seal swims fast . "
    "the fish swims in the sea ."
).split()

# For each word, count which words follow it in the training text.
follows = defaultdict(Counter)
for a, b in zip(training_text[:-1], training_text[1:]):
    follows[a][b] += 1

def next_word(word):
    options = follows[word]
    return random.choices(list(options), weights=list(options.values()))[0]

word = "the"
sentence = [word]
for _ in range(10):
    word = next_word(word)
    sentence.append(word)

print(" ".join(sentence))

Notice:
- It produces *plausible* sentences it never saw exactly by recombining patterns.
- It has no idea what a penguin *is*. It only knows what words tend to go together.
- Run it again and you may get something different (see "temperature" below).

## A few terms you will hear

- **Tokens** — models don't read whole words, but chunks ("token", "isation").
  Billing and length limits are counted in tokens.
- **Context window** — how much text the model can "see" at once. Go past it and
  it *forgets* the start of the conversation. This is why long chats drift.
- **Temperature / sampling** — the model picks among likely next-tokens with some
  randomness. Higher = more creative and more erratic; lower = more repetitive.
  A direct consequence: **the same prompt can give different answers.**
- **Transformer** — the neural-network design (from 2017) behind modern LLMs. Its
  key trick, *attention*, lets the model weigh which earlier words matter most for
  predicting the next one. You do not need the math, just the intuition: it is
  very good at using context.

## The one consequence to remember

Because an LLM generates *plausible-looking text*, it also generates
**plausible-looking code** and **plausible-looking function names**, whether or
not they are correct or even real.

That is not a bug you can prompt away. It is how the tool works. It is the reason
the rest of this lecture is about **verifying**, not trusting.

# 3. Chat assistants vs. agents

| | **Chat assistant** | **Agent** |
|---|---|---|
| Examples | openwebui, ChatGPT website | Claude Code, GitHub Copilot agents |
| What it does | text in → text out | can *run* commands, read/write files, see errors, and loop |
| Who runs the code | **you**, by copy-paste | the tool runs it itself |
| Sees your files/errors | only if you paste them in | yes, directly |

- In **today's exercise** you use the university chat assistant at
  [openwebui.uni-freiburg.de](https://openwebui.uni-freiburg.de). It only produces
  text *you* run and check everything.
- Later I'll demo an **agent** (Claude Code) that can run code and fix its own
  mistakes. It feels magical, but the same rule applies: you still verify.
- "**Agentic**" just means the AI can take actions in a loop (run → observe →
  decide → run again), not only emit one block of text.

# 4. The human-in-the-loop workflow

The reliable loop, whether chat or agent:

**specify → generate → read → run → test → iterate**

1. **Specify** — say exactly what you want (inputs, outputs, an example).
2. **Generate** — let the assistant write it.
3. **Read** — does the code actually do what you asked?
4. **Run** — execute it. Errors are information, not failure.
5. **Test** — check it against a case where *you* know the answer.
6. **Iterate** — paste back errors or wrong results; ask for a fix.

## Writing a good prompt

Decomposition (from earlier lectures) is your best prompting tool: ask for one clear
piece at a time.

**Weak prompt**

> write code to analyse my data

**Strong prompt**

> Write a Python function `mean_mass(rows)` using only the standard library.
> `rows` is a list of strings like `"Adelie;3200"`. Return the average of the
> second field as a float. Ignore rows where the value is `-999`. Show me an
> example call.

The strong prompt names the inputs, the output, the constraints (standard
library), an edge case (`-999`), and asks for an example. You will get code you
can actually check.

Other high-value moves:
- "**Explain** this line by line." (great for learning and for catching nonsense)
- "Now **write tests** / `assert` statements for this function."
- Paste the **exact error message** back. It is the single most useful thing you
  can give the assistant.

# 5. Failure modes

Three ways generated code goes wrong, from easiest to hardest to notice.

## 5a. Hallucinated APIs

The model invents a function or argument that *sounds* right but does not exist.
It read many programs and produced a plausible name, although one which happens to be wrong.

In [ ]:
import statistics

# The assistant confidently suggested statistics.average(...). Let's run it.
try:
    result = statistics.average([1, 2, 3])
except AttributeError as e:
    print("Crash:", e)

# The real function is statistics.mean:
print("Correct:", statistics.mean([1, 2, 3]))

**How you catch it:** you *run* it, and it crashes. Hallucinated APIs are an easy failure mode because the computer complains loudly.

## 5b. Plausible-but-wrong logic

The code runs without error and returns something reasonable-looking, but the
logic is subtly wrong. Read this function: it should return the *fraction* of
values above a threshold.

In [ ]:
def proportion_above(values, threshold):
    count = sum(1 for v in values if v > threshold)
    return count / threshold          # <-- looks fine at a glance

# Of [1,2,3,4,5], two values (4 and 5) are above 3, so the answer should be 2/5 = 0.4
print(proportion_above([1, 2, 3, 4, 5], 3))

It prints `0.666...`, not `0.4`. The bug: it divides by `threshold` instead of
by the number of values. No error, just a wrong number.

**How you catch it:** test against a case where you already know the answer.

## 5c. Silent data errors

The most dangerous. The code is correct *in general*, but wrong for *your data*,
and nothing complains. Real datasets often mark missing values with codes like
`-999`. Compute a mean without noticing, and you get a plausible, wrong number.

In [ ]:
# Penguin body masses in grams. -999 means "not measured" (a real convention).
masses = [3200, 4500, 3800, -999, 4100, -999, 3900]

naive_mean   = sum(masses) / len(masses)
clean        = [m for m in masses if m != -999]
correct_mean = sum(clean) / len(clean)

print(f"naive mean:   {naive_mean:.0f} g   <- no error, but wrong")
print(f"correct mean: {correct_mean:.0f} g")

Both numbers look like a penguin could weigh that much. Only one is right.

**How you catch it:** *look at your data* (print the extremes, sort it), and check
results against something you can compute by hand. **You will meet this exact trap
with real penguin data in today's exercise.**

# 6. Verifying generated code

Before you trust (or submit) any generated code:

- **Read** it. If you can't explain it, you can't trust it.
- **Run** it. Use inputs whose answer you already know.
- **Test edge cases**: empty input, missing values, negative numbers, duplicates.
- Ask the assistant to **write `assert` tests**, then read the tests critically too.
- For data: **look at the raw file first**, check the number of rows, the ranges,
  the units.
- Prefer **small, checkable pieces** over one giant block you can't inspect.

# 7. Reproducibility, provenance, and academic integrity

LLMs are **non-deterministic** (remember temperature): the same prompt may give
different code tomorrow. So for work that must be reproducible:

- **Record what you did**: which model/tool, and the key prompts you used.
- **Provenance of data**: where did the data come from? What do the columns mean?
  What are the missing-value codes? (See 5c.)
- **You own what you submit.** Course policy: you must be able to *understand,
  test, and explain* any code you hand in, regardless of who or what wrote it.
- Using AI is **encouraged and allowed** here. Hiding it, or submitting code you
  cannot explain, is not. Some exercises ask you to document how you used AI.
- Good scientific practice = the AI is a tool you can account for, like any
  instrument in the lab.

# 8. Live demo: Claude Code doing the heavy lifting

*(Watch. You do not need to use the tool used here; the point is to see the
ceiling of what agents can do, and that verification still matters.)*

An **agent** (here, Claude Code running a strong model) can take a bigger task and
run the loop *itself*: write code → run it → read the error → fix it → repeat.

What to watch for during the demo:
- It **runs** the code and reads real error messages, then corrects itself.
- It can **write and run tests** to check its own work.
- It is fast and capable. Still, it is sometimes confidently wrong.
- My job on screen is the same as yours in the exercise: **read, run, verify.**

Contrast: your chat assistant can only *describe* code; it cannot run it. That
gap is the whole difference between "chat" and "agent".

# 9. Today's exercise

You will use the university chat assistant to:

- get a small function written and **verify it with tests**,
- load a **real, messy** penguin dataset (odd separators, a decimal comma) despite initial errors,
- compute a mean and discover a **silent** missing-data trap (the `-999` from 5c),
- **fix three broken functions** an assistant might have written (one per failure
  mode above),
- make a plot and verify the data behind it,
- and document which model you used and one mistake you caught.

The goal is not just working code. The goal is code you can **stand behind**.